# 03 — g_1 Triplet Construction (Step A)

**Primary author:** Victoria

**Builds on:**
- *archive/09_learned_g_misdirection.ipynb* (Nathan — triplet design, training loop, concept-aligned extraction)
- *01_wn_filtering_and_split.ipynb* (Victoria — `clues_wn_filtered.csv` and split assignments)
- *02_phrase_construction_wn.ipynb* (Victoria — `f_clue.csv` anchor phrases and `f_common_wndef.csv` positive/negative phrases)
- *Milestone II NB 05* (Hans — cosine-similarity top-100 distractors in `dataset_harder.parquet`)

**Prompt engineering:** Victoria  
**AI assistance:** Claude / Claude Code (Anthropic)  
**Environment:** Local

---

This notebook is **Step A** of the design document (§6.5): a faithful
reproduction of NB 09's `T_1` triplet design on our new pipeline's data
artifacts. The output is a single committed CSV (`data/triplets/g1.csv`) that
the Great Lakes training script (`scripts/train_g1.py`) reads to fine-tune
g_stock into g_1.

The triplet design is (Anchor, Positive, Negative) where:

- **Anchor** = `f_clue(definition)` — the clue surface with `<t></t>` delimiters
  wrapping the definition span. This is the "treatment" phrase — what the
  model sees when the misleading clue context is present.
- **Positive** = `f_common_wndef(answer_wn)` — the answer word decontextualized
  as `"<t>word</t>: WordNet definition"`.
- **Negative** = `f_common_wndef(distractor_wn)` — a cosine-similarity-harder
  distractor word (from Milestone II NB 05), in the same decontextualized form.

The construction is entirely a join: our committed phrase files cover the
anchor, positive, and negative text, and `dataset_harder.parquet` supplies
the (clue_id, definition_wn) → distractor_wn mapping. All heavy lifting was
done upstream; this notebook is a five-minute CPU task that produces a
compact, inspectable artifact.

**Why a reproduction rather than a new design?** NB 09 found that training
on T_1 made the ATE *more* negative — the model compressed `f_common_wndef`
format embeddings into a cluster, artificially lowering the decontextualized
baseline similarity. Confirming this failure pattern under our cleaner
pipeline is a prerequisite for Step B (diagnosis) and Step C (an improved
triplet design).

**Reads:**
- `data/filtered_split/wn_synset/clues_wn_filtered.csv`
- `data/filtered_split/wn_synset/clue_phrases/f_clue.csv`
- `data/filtered_split/wn_synset/wndef/f_common_wndef.csv`
- `../../clue_misdirection/data/dataset_harder.parquet`

**Writes:**
- `data/triplets/g1.csv` — one row per training triplet
- `data/triplets/g1_meta.json` — provenance metadata (which f for each role, source paths, row counts)
- `outputs/03_train_g1-results.md` — coverage statistics and comparison to NB 09

---

## §0 — Imports and configuration

Environment auto-detection lets this notebook run unmodified on Local, Great
Lakes, and Colab. This is a CPU-only notebook — `pandas`, `numpy`, `json`, and
`pyarrow` (implicit, for parquet) are the only dependencies.

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================
import json
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    PROJECT_ROOT = Path("../..").resolve()

COMPONENT_ROOT = PROJECT_ROOT / "custom_embedding_model"
WN_DIR         = COMPONENT_ROOT / "data" / "filtered_split" / "wn_synset"
TRIPLET_DIR    = COMPONENT_ROOT / "data" / "triplets"
OUTPUT_DIR     = COMPONENT_ROOT / "outputs"

TRIPLET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# dataset_harder.parquet lives in the sibling clue_misdirection/ component
# (it was produced by Milestone II NB 05 and we treat it as a read-only input).
HARDER_PATH = PROJECT_ROOT / "clue_misdirection" / "data" / "dataset_harder.parquet"

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment:    {env_label}")
print(f"PROJECT_ROOT:   {PROJECT_ROOT}")
print(f"WN_DIR:         {WN_DIR}")
print(f"TRIPLET_DIR:    {TRIPLET_DIR}")
print(f"HARDER_PATH:    {HARDER_PATH}")

# Version reporting (Decision 18) — makes environment mismatches visible
# at a glance when a collaborator re-runs the notebook.
print(f"\npandas:  {pd.__version__}")
print(f"numpy:   {np.__version__}")

---

## §1 — Load source data

Four inputs flow into the triplet construction:

1. **`clues_wn_filtered.csv`** — the canonical clue file with split assignments.
   We restrict to `split == 'train'` immediately; all downstream operations
   touch only training-split rows (Decision 9: test-set lockout).
2. **`f_clue.csv`** — anchor phrases, indexed by `(clue_id, definition)`. Has
   full coverage over `clues_wn_filtered` (NB 02 confirmed 239,406/239,406).
3. **`f_common_wndef.csv`** — positive and negative phrases, indexed by `word`.
   Full coverage over the wn_synset vocabulary (every word has ≥1 synset by
   the NB 01 filter).
4. **`dataset_harder.parquet`** — the Milestone II artifact that supplies
   cosine-similarity top-100 distractors. We keep only the `label==0` rows
   (distractor pairs) and extract the `(clue_id, definition_wn)` →
   `distractor_wn` mapping.

Each loader uses `keep_default_na=False, na_values=[""]` on the CSVs because
"nan" is a valid crossword word (grandmother); pandas's default NA handling
would silently convert it to `NaN`.

In [ ]:
# ============================================================
# Load inputs
# ============================================================
t0 = time.time()

# --- 1. Canonical clue file (restrict to training split immediately) ---
clues_wn = pd.read_csv(
    WN_DIR / "clues_wn_filtered.csv",
    keep_default_na=False, na_values=[""],  # "nan" is a valid crossword word
)
wn_train = clues_wn[clues_wn["split"] == "train"].copy()

# --- 2. f_clue anchor phrases ---
f_clue = pd.read_csv(
    WN_DIR / "clue_phrases" / "f_clue.csv",
    keep_default_na=False, na_values=[""],
)
# Build a dict keyed by (clue_id, definition) — O(1) lookup per training row
f_clue_lookup = dict(
    zip(zip(f_clue["clue_id"], f_clue["definition"]), f_clue["phrase"])
)

# --- 3. f_common_wndef positive/negative phrases ---
f_wndef = pd.read_csv(
    WN_DIR / "wndef" / "f_common_wndef.csv",
    keep_default_na=False, na_values=[""],
)
f_wndef_lookup = dict(zip(f_wndef["word"], f_wndef["phrase"]))

# --- 4. dataset_harder distractor assignments ---
harder = pd.read_parquet(HARDER_PATH)
# label==0 rows carry the distractor word in answer_wn (the clue's real answer
# is in `answer`, and the real answer_wn is on the paired label==1 row).
dist = harder.loc[harder["label"] == 0, ["clue_id", "definition_wn", "answer_wn"]].copy()
dist = dist.rename(columns={"answer_wn": "distractor_wn"})

elapsed = time.time() - t0

print(f"clues_wn_filtered.csv:         {len(clues_wn):,} rows (all splits)")
print(f"  training split:              {len(wn_train):,} rows")
print(f"f_clue.csv:                    {len(f_clue):,} anchor phrases")
print(f"f_common_wndef.csv:            {len(f_wndef):,} positive/negative phrases")
print(f"dataset_harder.parquet:        {len(harder):,} rows (label=0 + label=1)")
print(f"  distractor rows (label=0):   {len(dist):,}")
print(f"\nLoad time: {elapsed:.1f}s")

# Expected-size checks — if these fail the upstream pipeline has drifted
# and the triplet file would be invalid.
assert len(f_clue) == 239_406, f"Expected 239,406 f_clue rows, got {len(f_clue):,}"
assert len(f_wndef) == 53_930, f"Expected 53,930 f_wndef rows, got {len(f_wndef):,}"

---

## §2 — Join training rows with distractors

The triplet's negative component depends on `dataset_harder.parquet`'s
pre-computed distractor assignments. We merge the training-split rows with
the distractor lookup on `(clue_id, definition_wn)`. Not every training row
has a matching distractor: `dataset_harder.parquet` was produced by the
Milestone II filtering pipeline, which applies slightly different upstream
constraints than our NB 01 WordNet filter. Rows that pass our filter but
were absent from the Milestone II dataset have no pre-computed distractor
and are lost to this inner join.

The expected loss is small (~2%). A larger loss would indicate a data
provenance issue worth investigating before training.

In [ ]:
# ============================================================
# Inner join: training rows ∩ harder distractors
# ============================================================
n_before = len(wn_train)

merged = wn_train.merge(dist, on=["clue_id", "definition_wn"], how="inner")

n_after = len(merged)
n_lost = n_before - n_after

print(f"Training rows before join:       {n_before:,}")
print(f"Training rows after join:        {n_after:,}")
print(f"Rows lost (no distractor):       {n_lost:,} ({n_lost/n_before:.1%})")
print(f"(Spec expected ~70,415 retained, ~1,692 lost / 2.3% — "
      f"small loss is normal: Milestone II's pipeline differs slightly from ours.)")

# The join is on (clue_id, definition_wn); verify the join did not silently
# duplicate training rows (which would happen if `dist` had duplicate keys).
n_dist_dup = dist.duplicated(subset=["clue_id", "definition_wn"]).sum()
print(f"\nDistractor key uniqueness check: {n_dist_dup} duplicate (clue_id, definition_wn) keys in harder")
assert n_after <= n_before + n_dist_dup, (
    f"Join inflated row count unexpectedly: {n_before:,} → {n_after:,}"
)

---

## §3 — Look up phrases for all three triplet roles

With the distractor attached to every surviving row, we now resolve the three
phrase strings:

- `anchor` from `f_clue_lookup[(clue_id, definition)]` — note the key uses the
  original-case `definition`, not `definition_wn`, because that's how
  `f_clue.csv` is indexed.
- `positive` from `f_wndef_lookup[answer_wn]`
- `negative` from `f_wndef_lookup[distractor_wn]`

Missing-lookup handling follows the strict-f convention (Decision 5): a row
with any missing phrase is dropped rather than silently filled. The only
meaningful loss expected here is on the `negative` side, because a handful
of distractor words come from Milestone II's wider vocabulary and are not in
our stricter `vocabulary_wndef.csv`. Anchor and positive should have 100%
coverage (f_clue covers all of `clues_wn_filtered`; wndef covers all of our
vocabulary).

In [ ]:
# ============================================================
# Resolve anchor / positive / negative phrases
# ============================================================
# Anchor: (clue_id, definition) composite key — definition is original case,
# matching f_clue.csv's indexing convention.
merged["anchor"] = [
    f_clue_lookup.get((cid, defn))
    for cid, defn in zip(merged["clue_id"], merged["definition"])
]
# Positive and negative: single-word keys on the wndef phrase lookup.
merged["positive"] = merged["answer_wn"].map(f_wndef_lookup)
merged["negative"] = merged["distractor_wn"].map(f_wndef_lookup)

# Count missing-lookup rows per role BEFORE dropping, so the loss is
# attributable to a specific phrase file.
n_pre_drop = len(merged)
miss_anchor   = merged["anchor"].isna().sum()
miss_positive = merged["positive"].isna().sum()
miss_negative = merged["negative"].isna().sum()

# Distractor words absent from our wndef vocabulary — the largest expected loss.
missing_distractors = set(
    merged.loc[merged["negative"].isna(), "distractor_wn"]
)

triplets = merged.dropna(subset=["anchor", "positive", "negative"]).copy()
n_post_drop = len(triplets)
n_dropped   = n_pre_drop - n_post_drop

print(f"Rows before phrase lookup:      {n_pre_drop:,}")
print(f"  missing anchor:               {miss_anchor:,}    (expected 0 — f_clue has full coverage)")
print(f"  missing positive:             {miss_positive:,}    (expected 0 — all answers in wndef vocab)")
print(f"  missing negative:             {miss_negative:,}    (expected ~494 — some distractors outside our vocab)")
print(f"Rows after dropping missing:    {n_post_drop:,}")
print(f"Rows dropped (any missing):     {n_dropped:,} ({n_dropped/n_pre_drop:.2%})")
print(f"Unique missing distractor words: {len(missing_distractors):,}")

---

## §4 — Build and save the triplet file

We keep only the columns the training script needs — `clue_id`, `definition`,
`answer_wn`, `distractor_wn`, plus the three phrase strings — and save to
`data/triplets/g1.csv`. The `split` column is deliberately dropped: a triplet
file is training-only by construction (DATA.md), and including `split` would
invite downstream code to re-filter on it, which could mask a bug.

Before saving we assert a small battery of schema invariants — any failure
here means the committed artifact is not what the spec describes.

In [ ]:
# ============================================================
# Assemble the triplet DataFrame and validate invariants
# ============================================================
g1 = triplets[[
    "clue_id", "definition", "answer_wn", "distractor_wn",
    "anchor", "positive", "negative",
]].reset_index(drop=True).copy()

# --- Invariant checks ---
assert g1.notna().all().all(), "Null values remain in g1 after dropna"
# Anchor must contain a single balanced <t>...</t> span.
assert (g1["anchor"].str.count("<t>") == 1).all(), "Some anchors missing/extra <t>"
assert (g1["anchor"].str.count("</t>") == 1).all(), "Some anchors missing/extra </t>"
# Positive and negative are f_common_wndef phrases — they always start with <t>.
assert g1["positive"].str.startswith("<t>").all(), "Some positives don't start with <t>"
assert g1["negative"].str.startswith("<t>").all(), "Some negatives don't start with <t>"
# A training signal where positive == negative has no meaning (the answer
# would be identical to the distractor for that row). Should be impossible
# given how dataset_harder was constructed, but worth enforcing.
n_pos_eq_neg = (g1["positive"] == g1["negative"]).sum()
assert n_pos_eq_neg == 0, f"Found {n_pos_eq_neg} rows where positive == negative"
# 'split' must NOT appear — a triplet file is training-only by definition.
assert "split" not in g1.columns, "split column leaked into triplet file"

print(f"All schema invariants passed.")
print(f"Final triplet rows: {len(g1):,}")

# --- Save ---
g1_path = TRIPLET_DIR / "g1.csv"
g1.to_csv(g1_path, index=False)

size_mb = g1_path.stat().st_size / (1024 * 1024)
print(f"Saved: {g1_path.relative_to(COMPONENT_ROOT)}  ({size_mb:.1f} MB)")

---

## §5 — Save provenance metadata

Per Decision 10, every triplet CSV has a companion `<g_name>_meta.json`
recording which f was used for each triplet role, the source paths of the
phrase files, and the resulting row counts. This lets a future reader
reconstruct exactly how a triplet file was built without reading this
notebook, and prevents silent drift if the upstream phrase files are
regenerated with different conventions.

In [ ]:
# ============================================================
# Write g1_meta.json
# ============================================================
meta = {
    "g_name": "g1",
    "triplet_design": "T_1",
    "description": "Step A reproduction of NB 09 triplet design on our pipeline's data artifacts",
    "anchor_f": "f_clue",
    "anchor_source": "data/filtered_split/wn_synset/clue_phrases/f_clue.csv",
    "positive_f": "f_common_wndef",
    "positive_source": "data/filtered_split/wn_synset/wndef/f_common_wndef.csv",
    "negative_f": "f_common_wndef",
    "negative_source": "data/filtered_split/wn_synset/wndef/f_common_wndef.csv",
    "distractor_source": "../../clue_misdirection/data/dataset_harder.parquet",
    "distractor_method": "cosine-similarity top-100 from Milestone II NB 05",
    "split": "train",
    "n_rows": int(len(g1)),
    "n_unique_clue_ids": int(g1["clue_id"].nunique()),
    "n_unique_pairs": int(g1[["definition", "answer_wn"]].drop_duplicates().shape[0]),
    "n_unique_answers": int(g1["answer_wn"].nunique()),
    "n_unique_distractors": int(g1["distractor_wn"].nunique()),
    "random_state": 42,
    "date_created": date.today().isoformat(),
    "rows_lost_to_distractor_join": int(n_lost),
    "rows_lost_to_missing_phrases": int(n_dropped),
    "unique_distractor_words_not_in_wndef_vocab": int(len(missing_distractors)),
}

meta_path = TRIPLET_DIR / "g1_meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Saved: {meta_path.relative_to(COMPONENT_ROOT)}")
print(json.dumps(meta, indent=2))

---

## §6 — Inspection and examples

Print five concrete triplets in full so Victoria (and any reviewer) can
visually confirm the construction before Nathan runs the training job on
Great Lakes. This is the cheapest bug-catcher available — if any phrase
looks malformed, it's much better to notice here than after a 3-hour GPU run.

In [ ]:
# ============================================================
# Show 5 example triplets in full
# ============================================================
for i, row in g1.head(5).iterrows():
    print(f"[Row {i}] clue_id={row['clue_id']} "
          f"definition='{row['definition']}' "
          f"answer_wn='{row['answer_wn']}' "
          f"distractor_wn='{row['distractor_wn']}'")
    print(f"  Anchor:   {row['anchor']}")
    print(f"  Positive: {row['positive']}")
    print(f"  Negative: {row['negative']}")
    print()

In [ ]:
# ============================================================
# Triplet-set statistics
# ============================================================
n_rows        = len(g1)
n_clues       = g1["clue_id"].nunique()
n_pairs       = g1[["definition", "answer_wn"]].drop_duplicates().shape[0]
n_answers     = g1["answer_wn"].nunique()
n_distractors = g1["distractor_wn"].nunique()
# A word that plays both roles across different rows would be a signal that
# the distractor pool overlaps with the answer vocabulary — expected, since
# distractors are drawn from the same word space.
n_overlap = len(set(g1["answer_wn"]) & set(g1["distractor_wn"]))

print(f"Total triplet rows:                         {n_rows:,}")
print(f"Unique clue_ids:                            {n_clues:,}")
print(f"Unique (definition, answer_wn) pairs:       {n_pairs:,}")
print(f"Unique answer words:                        {n_answers:,}")
print(f"Unique distractor words:                    {n_distractors:,}")
print(f"Words appearing as BOTH answer & distractor")
print(f"  (across different rows):                  {n_overlap:,}")

---

## §7 — Comparison to NB 09

NB 09's published training run used `SAMPLE_MODE=True`, which randomly
subsampled 20,000 unique (definition, answer) pairs from the 102,086 available
training pairs — yielding 37,593 training triplet rows. Its unsampled training
set would have been 192,039 rows.

Our pipeline produces a training triplet set **between** those two sizes: the
smaller total is a direct consequence of Decision 3's 30/20/50 train/validate/
test split (vs. NB 09's 80/20), which is an intentional investment in final
evaluation credibility (Decision 9). The triplet design itself — anchor,
positive, and negative roles and their phrase constructions — matches NB 09
exactly.

In [ ]:
# ============================================================
# NB 09 comparison table
# ============================================================
nb09_full_rows  = 192_039
nb09_full_pairs = 102_086
nb09_samp_rows  = 37_593
nb09_samp_pairs = 20_000

print(f"{'Dataset':<30s} {'Pairs':>10s} {'Triplet rows':>14s}")
print("-" * 58)
print(f"{'NB 09 full (unsampled)':<30s} {nb09_full_pairs:>10,} {nb09_full_rows:>14,}")
print(f"{'NB 09 sampled (published)':<30s} {nb09_samp_pairs:>10,} {nb09_samp_rows:>14,}")
print(f"{'g_1 (this notebook)':<30s} {n_pairs:>10,} {n_rows:>14,}")
print()
print(f"g_1 / NB 09 sampled:  {n_rows / nb09_samp_rows:.2f}x triplet rows")
print(f"g_1 / NB 09 full:     {n_rows / nb09_full_rows:.2f}x triplet rows")

---

## §8 — Summary

This notebook produced `data/triplets/g1.csv` and its metadata companion.
The next action is to upload `g1.csv` to Great Lakes and submit
`scripts/train_g1.sh` to run the fine-tuning job. After training, model
weights go to Google Drive (Decision 12) and a `README.md` is filled in from
the SLURM log's SUMMARY block.

**Coverage log**
- Training rows from `clues_wn_filtered.csv`: **72,107**
- After inner join with `dataset_harder.parquet` distractors: ~70,415
  (small loss because Milestone II's pipeline and our NB 01 filter
  disagree on a thin margin of rows)
- After dropping rows where any phrase lookup failed: ~69,921
  (the only meaningful loss is on `negative`, due to distractor words outside
  `vocabulary_wndef.csv`)

**Outputs**
- `data/triplets/g1.csv`
- `data/triplets/g1_meta.json`
- `outputs/03_train_g1-results.md` (written in §9)

**Key observation** — this is a faithful T_1 reproduction. The only
differences from NB 09 are (a) upstream filtering (our NB 01 WordNet filter
vs. Milestone II's filtering), and (b) the split fraction (30/20/50 vs.
80/20). Everything about the triplet design, phrase construction, and
planned training hyperparameters matches NB 09. A runtime on the order of a
few seconds for this entire notebook reflects the fact that all heavy
lifting was done in NB 01 and NB 02.

---

## §9 — Write results file

Per the component CLAUDE.md, every notebook writes a concise results file
to `outputs/<notebook-name>-results.md`. This is the artifact the Architect
reviews; it also becomes the source of truth for the FINDINGS.md entry on
Stage 3.

In [ ]:
# ============================================================
# Write outputs/03_train_g1-results.md
# ============================================================
g1_bytes = g1_path.stat().st_size
meta_bytes = meta_path.stat().st_size

lines = []
lines.append("# Results: 03 — g_1 Triplet Construction (Step A)")
lines.append("")
lines.append("## Versions")
lines.append("")
lines.append(f"- pandas: {pd.__version__}")
lines.append(f"- numpy:  {np.__version__}")
lines.append("")
lines.append("## Input files")
lines.append("")
lines.append(f"- `clues_wn_filtered.csv`: {len(clues_wn):,} rows total, "
             f"{len(wn_train):,} in training split")
lines.append(f"- `f_clue.csv`: {len(f_clue):,} anchor phrases (full wn_synset scope)")
lines.append(f"- `f_common_wndef.csv`: {len(f_wndef):,} positive/negative phrases")
lines.append(f"- `dataset_harder.parquet`: {len(harder):,} rows total, "
             f"{len(dist):,} label=0 distractor rows")
lines.append("")
lines.append("## Join and coverage statistics")
lines.append("")
lines.append(f"- Training rows before distractor join: {n_before:,}")
lines.append(f"- Training rows after distractor join: {n_after:,} "
             f"({n_after/n_before:.2%} retained)")
lines.append(f"- Rows lost to distractor join: {n_lost:,} ({n_lost/n_before:.2%})")
lines.append(f"- Rows lost to missing anchor phrase: {int(miss_anchor):,}")
lines.append(f"- Rows lost to missing positive phrase: {int(miss_positive):,}")
lines.append(f"- Rows lost to missing negative phrase: {int(miss_negative):,}")
lines.append(f"- Unique distractor words absent from `vocabulary_wndef.csv`: "
             f"{len(missing_distractors):,}")
lines.append(f"- Final triplet rows: {n_rows:,}")
lines.append("")
lines.append("## Triplet-set structure")
lines.append("")
lines.append(f"- Unique clue_ids: {n_clues:,}")
lines.append(f"- Unique (definition, answer_wn) pairs: {n_pairs:,}")
lines.append(f"- Unique answer words: {n_answers:,}")
lines.append(f"- Unique distractor words: {n_distractors:,}")
lines.append(f"- Words appearing as both answer and distractor "
             f"(across different rows): {n_overlap:,}")
lines.append("")
lines.append("## Comparison to NB 09")
lines.append("")
lines.append("| Dataset | Pairs | Triplet rows |")
lines.append("|---|---:|---:|")
lines.append(f"| NB 09 full (unsampled) | {nb09_full_pairs:,} | {nb09_full_rows:,} |")
lines.append(f"| NB 09 sampled (published) | {nb09_samp_pairs:,} | {nb09_samp_rows:,} |")
lines.append(f"| g_1 (this notebook) | {n_pairs:,} | {n_rows:,} |")
lines.append("")
lines.append(f"- g_1 / NB 09 sampled ratio: {n_rows / nb09_samp_rows:.2f}×")
lines.append(f"- g_1 / NB 09 full ratio: {n_rows / nb09_full_rows:.2f}×")
lines.append("")
lines.append("## Outputs")
lines.append("")
lines.append(f"- `data/triplets/g1.csv` — {g1_bytes/1024/1024:.1f} MB, {n_rows:,} rows")
lines.append(f"- `data/triplets/g1_meta.json` — {meta_bytes/1024:.1f} KB")
lines.append("")
lines.append("## Runtime")
lines.append("")
lines.append(f"- Input load: {elapsed:.1f}s")
lines.append("- Notebook end-to-end: a few seconds (CPU, text manipulation only)")
lines.append("")
lines.append("## Interpretation")
lines.append("")
lines.append("This is a faithful reproduction of NB 09's T_1 triplet design on our "
             "pipeline's cleaner data. The only material differences from NB 09 are "
             "upstream filtering (our NB 01 WordNet filter vs. Milestone II's pipeline) "
             "and split proportions (30/20/50 vs. 80/20, per Decision 3). The anchor, "
             "positive, and negative roles and their phrase constructions match NB 09 "
             "exactly — so if the T_1 failure pattern (format-specific compression of "
             "f_common_wndef phrases) recurs here, it is attributable to the triplet "
             "design itself and not to any data-preparation artifact.")
lines.append("")

results_path = OUTPUT_DIR / "03_train_g1-results.md"
results_path.write_text("\n".join(lines))
print(f"Saved: {results_path.relative_to(COMPONENT_ROOT)}")